# 🫀 실험 3 — 형태 축 vs 리듬 축: **부정맥 밖으로 나간다**

**MedKOS / `notebooks/exp3_ptbxl_morph_vs_rhythm.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## 왜 이 실험인가 — 실험1′의 거울상

실험1′에서 얻은 것:

> **S(심방조기박동)는 형태로는 안 잡히고 리듬으로 잡힌다.**
> B2(형태만) S F1 0.306 → B3(+RR) 0.445 · B3−B2 Δ+0.1268 [+0.081,+0.176] 유의

그런데 이건 **부정맥 하나에서만 확인한 결론**입니다. 반대 방향은 아직 안 봤습니다:

> **전도장애(LBBB/RBBB)와 ST-T 변화는 리듬으로는 안 잡히고 형태로 잡히는가?**

전도장애는 **조기성이 없습니다.** 정상 심박수, 정상 RR인데 QRS만 넓습니다.
ST-T 변화도 마찬가지로 리듬과 무관합니다. 그러니 **리듬 축만 준 모델은 거의 무력해야
정상**이고, 형태 축이 이겨야 합니다.

**두 방향이 다 확인되면 "형태 축과 리듬 축은 서로 다른 질환군을 담당한다"가 양방향으로
증명됩니다.** 그게 이 프로젝트 구조(⑥ Finding Head의 축 분리)의 실증적 근거가 됩니다.

## 그리고 — 목표가 부정맥이 아니라 심장 질환이므로

| | 실험 1′·2 | **실험 3** |
|---|---|---|
| 데이터 | Icentia11k (웨어러블 단일유도) | **PTB-XL** (임상 12유도 진단 데이터) |
| 라벨 | 비트 기원 (N/S/V) | **진단명** (NORM / CD / STTC) |
| 단위 | 비트 | **레코드(환자)** |
| 축 | 부정맥 | **전도장애 · 허혈성 ST-T** |

**부정맥 비트 분류에서 진단 분류로 넘어가는 첫 실험**입니다. 그리고 PTB-XL은 12유도라
**실험 10(유도 ablation)의 발판**이 그대로 만들어집니다.

---

## 라벨-유도 정합성 필터 (퀘스트의 실험0을 여기서 실천)

PTB-XL 5개 superclass 중 **lead II에서 관측 가능한 것만** 씁니다
(`ailab-2026-0016` 2-bis 표):

| superclass | lead II 관측 | 이번 실험 |
|---|---|---|
| **NORM** 정상 | ✅ | **사용** |
| **CD** 전도장애 | ✅ QRS 폭 (좌우 구분은 불가하나 '넓다'는 관측 가능) | **사용** |
| **STTC** ST-T 변화 | ✅ ST 편위·T 역위 | **사용** |
| MI 심근경색 | ❌ 국소화가 흉부유도 필요 | **제외** |
| HYP 비대 | ❌ 전압 기준이 흉부유도 기반 | **제외** |

**MI·HYP를 lead II로 맞히라고 하면 모델이 환각을 학습합니다.** 이 필터가 그걸 막습니다.

## 사전등록

```
주가설 : (형태) − (리듬) > 0        ← 실험1′과 반대 방향 예측
비교   : M − R  (형태만 vs 리듬만)   ★ 주가설
         MR − M (리듬을 더하면 오르나) → 오르지 않아야 예측 부합
지표   : 테스트 fold의 macro-F1 (NORM/CD/STTC)
분할   : PTB-XL 공식 strat_fold — 1~8 학습 / 9 검증 / 10 테스트 (환자 계층화 완료)
저울   : seed 3개 확률 평균
판정   : 부트스트랩(레코드 단위) 95% CI가 0을 벗어나는가
금지   : fold 10을 보고 아무것도 바꾸지 않는다
```

**예측을 미리 적어둡니다** — 실험1′의 대칭이 성립한다면:
`M − R` 은 **크게 양수**, `MR − M` 은 **0 근처**여야 합니다.
반대가 나오면 "형태/리듬 이분법"이라는 전제 자체를 재고해야 합니다.


In [ ]:
# CELL 1 — Drive lib 재사용 + 설정
!pip -q install wfdb neurokit2

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

QUICK   = True        # True: 레코드 6000개로 빠르게 / False: 전체
FS      = 100         # PTB-XL records100 (공식 벤치마크가 쓰는 해상도)
LEAD    = "II"
CLASSES = ["NORM", "CD", "STTC"]     # ★ MI·HYP는 lead II 관측 불가라 제외
N_SEEDS, EPOCHS = (2, 15) if QUICK else (3, 25)
SEED0 = 20260731

CONFIG = dict(exp="exp3_ptbxl_morph_vs_rhythm", quest="ailab-2026-0015",
              hypothesis="(형태) − (리듬) > 0 — 전도장애·ST-T는 형태 축이 담당하는가",
              prediction="M−R 크게 양수, MR−M 0 근처 (실험1′의 대칭)",
              dataset="ptb-xl 1.0.3 records100", lead=LEAD, classes=CLASSES,
              excluded_labels=["MI", "HYP"], exclusion_reason="lead II 관측 불가",
              split="official strat_fold 1-8/9/10", fs=FS,
              n_seeds=N_SEEDS, epochs=EPOCHS, seed0=SEED0, quick=QUICK, boot=2000)
np.random.seed(SEED0)
# 그림 한글 깨짐 방지 (Colab)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"],
                       capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as _e:
    print("한글 폰트 설정 생략:", _e)

run = MedKOSRun("exp3_ptbxl", CONFIG, project=PROJECT)
run.log(f"QUICK={QUICK} · classes={CLASSES} · seed {N_SEEDS} · epoch {EPOCHS}")

### CELL 2 — PTB-XL 내려받기 (★ 여기서 한 번 멈추세요)

`records100`만 쓰면 약 **250 MB**입니다(전체 zip 1.7 GB 대신). Drive에 캐시하므로 한 번만
받습니다. 구조를 확인하고 다음으로 넘어가세요.

In [ ]:
# CELL 2 — PTB-XL 확보 + 구조 확인 (셸 매직 대신 순수 Python)
import wfdb, pandas as pd, subprocess, zipfile, shutil

# 신호는 /content(로컬 디스크)에 푼다 — Drive에 작은 파일 2만 개를 쓰면 매우 느리다.
# Drive에는 최종 npz 하나만 남긴다(CELL 4).
PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
ZIP = ("https://physionet.org/static/published-projects/ptb-xl/"
       "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip")

def fetch(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    r = subprocess.run(["wget", "-q", "-O", dest, url])
    return r.returncode == 0 and os.path.getsize(dest) > 0

for f in ("ptbxl_database.csv", "scp_statements.csv"):
    ok = fetch(f"{BASE}/{f}", os.path.join(PTB, f))
    run.log(f"{'✅' if ok else '❌'} {f}")

df = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")
scp = pd.read_csv(os.path.join(PTB, "scp_statements.csv"), index_col=0)
run.log(f"레코드 {len(df):,} · 환자 {df.patient_id.nunique():,} · "
        f"fold {sorted(df.strat_fold.unique())}")
run.log(f"진단클래스: {sorted(scp.diagnostic_class.dropna().unique())}")

# 신호: 전체 zip 한 번(약 1.7GB, 5~15분). 작은 파일을 2만 번 받는 것보다 훨씬 빠르다.
NPZ_DONE = os.path.exists(run.data(f"ptbxl_leadII_{'quick' if QUICK else 'full'}.npz"))
if NPZ_DONE:
    run.log("최종 npz 캐시가 이미 있어 신호 다운로드를 건너뜁니다")
elif not os.path.isdir(os.path.join(PTB, "records100")):
    zp = "/content/ptbxl.zip"
    run.log("PTB-XL zip 내려받는 중 (약 1.7GB, 5~15분)…")
    if fetch(ZIP, zp):
        run.log("압축 푸는 중…")
        with zipfile.ZipFile(zp) as z:
            for n in z.namelist():
                if "/records100/" in n or n.endswith(".csv"):
                    z.extract(n, "/content/_ptb")
        # 최상위 폴더 한 겹 벗기기
        top = [d for d in os.listdir("/content/_ptb") if os.path.isdir(f"/content/_ptb/{d}")]
        if top:
            src = f"/content/_ptb/{top[0]}"
            for item in os.listdir(src):
                dst = os.path.join(PTB, item)
                if not os.path.exists(dst):
                    shutil.move(os.path.join(src, item), dst)
        os.remove(zp)
    else:
        run.log("⚠️ zip 실패 → wfdb 스트리밍으로 대체합니다(느림). PTB=None 으로 표시")
        PTB = None

run.log(f"records100 존재: "
        f"{PTB is not None and os.path.isdir(os.path.join(PTB, 'records100'))}")

# 한 레코드 확인
row = df.iloc[0]
try:
    if PTB and os.path.isdir(os.path.join(PTB, "records100")):
        r0 = wfdb.rdrecord(os.path.join(PTB, row.filename_lr))
    else:
        r0 = wfdb.rdrecord(row.filename_lr.split("/")[-1],
                           pn_dir="ptb-xl/1.0.3/" + "/".join(row.filename_lr.split("/")[:-1]))
    run.log(f"\n샘플 fs={r0.fs} sig={r0.sig_name} shape={r0.p_signal.shape}")
    run.save_json("ptbxl_probe", {"n_records": int(len(df)), "fs": int(r0.fs),
                                  "sig_name": list(r0.sig_name),
                                  "shape": list(r0.p_signal.shape), "local": bool(PTB)})
except Exception as e:
    run.log(f"❌ 샘플 레코드 읽기 실패: {type(e).__name__}: {str(e)[:120]}")
run.log("\n확인: ① fs=100 ② sig_name에 'II' ③ shape=(1000,12)")

In [ ]:
# CELL 3 — 라벨 만들기: superclass 단일라벨 + 유도 정합성 필터
agg = scp[scp.diagnostic == 1].diagnostic_class.to_dict()

def superclasses(codes_str):
    d = ast.literal_eval(codes_str)
    return sorted({agg[k] for k in d if k in agg})

df["sc"] = df.scp_codes.apply(superclasses)
df["n_sc"] = df.sc.apply(len)

# ★ 유도 정합성: MI·HYP가 하나라도 붙은 레코드는 제외(lead II로 관측 불가)
excl = df.sc.apply(lambda s: any(x in ("MI", "HYP") for x in s))
# 다중라벨은 해석이 흐려지므로 단일 superclass만 사용
# ★ pandas의 & 는 단축평가를 하지 않는다 — n_sc==1 필터가 걸리기 전에 람다가
#   전 행에서 실행되므로, 진단 코드가 하나도 없는 레코드(빈 리스트)에서 s[0]이 터진다.
#   PTB-XL에는 리듬/형태 코드만 붙은 레코드가 상당수 있다.
keep = (~excl) & df.sc.apply(lambda s: len(s) == 1 and s[0] in CLASSES)
sub = df[keep].copy()
sub["y"] = sub.sc.apply(lambda s: CLASSES.index(s[0]))

run.log(f"전체 {len(df):,}건")
run.log(f"  진단 superclass 없음(리듬/형태 코드만) {int((df.n_sc==0).sum()):,}건")
run.log(f"  MI 또는 HYP 포함(lead II 관측 불가)   {int(excl.sum()):,}건")
run.log(f"  다중 superclass(해석 모호)            {int((df.n_sc>1).sum()):,}건")
run.log(f"→ 단일 superclass {CLASSES} 만 남김 → **{len(sub):,}건**")
for i, c in enumerate(CLASSES):
    m = sub.y == i
    run.log(f"  {c:5s}: {int(m.sum()):6,}건 · 환자 {sub[m].patient_id.nunique():,}명")
run.log(f"fold 분포: {sub.strat_fold.value_counts().sort_index().to_dict()}")

if QUICK:
    sub = pd.concat([g.sample(min(len(g), 2000), random_state=SEED0)
                     for _, g in sub.groupby("y")])
    run.log(f"QUICK: 클래스별 최대 2000건 샘플 → {len(sub):,}건")

run.save_json("label_stats", {
    "n_total": int(len(df)), "n_excluded_MI_HYP": int(excl.sum()),
    "n_used": int(len(sub)),
    "per_class": {c: int((sub.y == i).sum()) for i, c in enumerate(CLASSES)}})

In [ ]:
# CELL 4 — 신호 로드(lead II) + 리듬 특징 추출
import neurokit2 as nk
from scipy.signal import find_peaks

CACHE = run.data(f"ptbxl_leadII_{'quick' if QUICK else 'full'}.npz")
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    X, R, Y, FOLD, PID = z["X"], z["R"], z["y"], z["fold"], z["pid"]
    run.log(f"캐시 재사용: {X.shape}")
else:
    def rr_features(sig):
        """10초 lead II → 리듬 요약 7개. R검출 실패 시 0벡터."""
        try:
            _, info = nk.ecg_peaks(sig, sampling_rate=FS)
            rp = np.asarray(info["ECG_R_Peaks"])
        except Exception:
            rp, _ = find_peaks(sig, distance=int(0.3 * FS), height=np.std(sig))
        if len(rp) < 4:
            return np.zeros(7, "float32"), 0
        rr = np.diff(rp) / FS
        med = np.median(rr)
        return np.array([
            med,                                   # 중앙 RR
            60.0 / max(med, 1e-3),                 # 심박수
            np.std(rr),                            # RR 산포
            np.sqrt(np.mean(np.diff(rr) ** 2)) if len(rr) > 1 else 0,   # RMSSD
            float(np.mean(np.abs(np.diff(rr)) > 0.05)) if len(rr) > 1 else 0,  # pNN50 유사
            rr.min() / (med + 1e-6),               # 최단/중앙 (조기성)
            rr.max() / (med + 1e-6),               # 최장/중앙 (휴지)
        ], "float32"), len(rp)

    Xs, Rs, Ys, Fs_, Ps, nofail = [], [], [], [], [], 0
    t0 = time.time()
    for k, (eid, row) in enumerate(sub.iterrows()):
        try:
            if PTB and os.path.isdir(os.path.join(PTB, "records100")):
                rec = wfdb.rdrecord(os.path.join(PTB, row.filename_lr))
            else:
                parts = row.filename_lr.split("/")
                rec = wfdb.rdrecord(parts[-1],
                                    pn_dir="ptb-xl/1.0.3/" + "/".join(parts[:-1]))
        except Exception:
            nofail += 1; continue
        names = [n.strip() for n in rec.sig_name]
        if LEAD not in names:
            nofail += 1; continue
        s = rec.p_signal[:, names.index(LEAD)].astype("float64")
        s = np.nan_to_num(s)
        q = np.percentile(s, 75) - np.percentile(s, 25)
        s = ((s - np.median(s)) / (q + 1e-6)).astype("float32")
        f, npk = rr_features(s)
        Xs.append(np.clip(s, -20, 20)); Rs.append(f); Ys.append(int(row.y))
        Fs_.append(int(row.strat_fold)); Ps.append(int(row.patient_id))
        if (k + 1) % 1000 == 0:
            run.log(f"  {k+1}/{len(sub)} ({time.time()-t0:.0f}s)")
    X = np.array(Xs, "float32"); R = np.array(Rs, "float32")
    Y = np.array(Ys); FOLD = np.array(Fs_); PID = np.array(Ps)
    np.savez_compressed(CACHE, X=X, R=R, y=Y, fold=FOLD, pid=PID)
    run.log(f"저장: {CACHE} (실패 {nofail}건)")

R = np.clip(np.nan_to_num(R), [0.2, 20, 0, 0, 0, 0, 0], [3, 220, 1, 1, 1, 2, 5])
run.log(f"\nX{X.shape} R{R.shape} · 클래스 {np.bincount(Y).tolist()}")
run.log(f"리듬특징 범위 min={R.min(0).round(2).tolist()}")
run.log(f"              max={R.max(0).round(2).tolist()}")

### CELL 5 — 사전점검: 리듬 특징만으로 세 클래스가 갈리는가

**예측: 거의 안 갈려야 정상입니다.** 전도장애·ST-T 변화는 심박수·RR과 무관하니까요.
여기서 리듬이 잘 갈린다면 라벨에 부정맥이 섞여 있다는 뜻이라 해석이 달라집니다.

In [ ]:
# CELL 5 — 리듬 축 사전점검 (예측: 판별력 낮아야 정상)
FEAT = ["RR중앙", "심박수", "RR산포", "RMSSD", "pNN50", "최단/중앙", "최장/중앙"]
pre = {}
run.log("리듬 특징별 NORM↔(CD·STTC) 분리 (Cohen's d)")
oth = Y != 0
for j, nm in enumerate(FEAT):
    a, b = R[~oth, j], R[oth, j]
    d = float((a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2 + 1e-9))
    pre[nm] = d
    run.log(f"  {nm:10s} d={d:+.3f}")
mx = max(abs(v) for v in pre.values())
run.log(f"\n최대 |d| = {mx:.3f}")
run.log("  예측대로면 0.3 미만 — 리듬은 이 질환군을 거의 못 가른다")
run.log("  0.5를 넘으면 라벨에 부정맥이 섞였을 가능성 → 해석 주의")
if 0.30 <= mx <= 0.50:
    run.log("  ⚠️ 0.3~0.5 회색지대 — CELL 5.5에서 원인(오염/중증도/검출)을 가르고 넘어갈 것")
run.save_json("precheck_rhythm", {"cohens_d": pre, "max_abs_d": mx})

### CELL 5.5 — 회색지대 진단: `|d|`가 0.3~0.5면 여기서 원인을 가른다

CELL 5의 사전등록 해석은 **0.3 미만이면 예측 적중 · 0.5 초과면 오염 의심**이었습니다.
그 사이가 나오면 셋 중 무엇인지 **학습 전에** 갈라둬야 나중에 결과를 해석할 수 있습니다.

| 가설 | 뜻 | 구분법 |
|---|---|---|
| **(a) 라벨 오염** | CD·STTC 레코드에 부정맥 리듬코드(AFIB 등)가 같이 붙어 있다 | 순수 SR 레코드만 남기면 `d`가 무너진다 |
| **(b) 중증도 대리** | 아픈 사람이 그냥 심박수가 빠르다 — 기전이 아니라 **비특이 지름길** | 순수 SR만 남겨도 `d`가 유지되고, 최대값이 심박수다 |
| **(c) 검출 아티팩트** | 넓은 QRS(CD)에서 R검출이 흔들려 '불규칙'이 인위적으로 생긴다 | 검출 실패율·빈맥률이 클래스별로 다르다 |

**PTB-XL의 리듬 코드는 진단 superclass와 별개 축이라, 지금까지 한 번도 거른 적이 없습니다.**
CELL 3의 필터는 `diagnostic` 코드만 봤습니다. (a)가 열려 있는 이유입니다.

> **중요 — 어느 쪽이든 CELL 6은 그대로 돌립니다.** (a)·(b)·(c) 전부 리듬 arm(R)을
> *실제 기전보다 좋게* 만드는 방향입니다. 즉 **M−R을 작게** 만듭니다. 그러니 M−R이
> 크게 양수로 나오면 이 오염을 **뚫고** 나온 것이라 결론이 더 강해지고, M−R이 0 근처면
> 그때 이 진단 결과를 꺼내 "R이 기전이 아니라 지름길로 벌었나"를 따집니다.


In [ ]:
# CELL 5.5 — 회색지대 진단 (학습 전 30초 · 다운로드 없음)
from collections import Counter

# 캐시가 다른 sub로 만들어졌으면 아래 분석이 통째로 거짓이 된다 — 먼저 정렬을 확인한다.
assert len(sub) == len(Y), f"정렬 불일치: sub {len(sub)} vs Y {len(Y)} — CELL 4를 캐시 없이 다시"
assert (sub.y.values == Y).all(), "라벨 순서 불일치 — 캐시가 다른 sub로 만들어졌다"

rhy_codes = set(scp[scp.rhythm == 1].index)
rl = [sorted(k for k in ast.literal_eval(s) if k in rhy_codes) for s in sub.scp_codes]
is_sr = np.array([ks == ["SR"] for ks in rl])          # 순수 정상동율동만
no_rhy = np.array([len(ks) == 0 for ks in rl])

run.log("── (a) 리듬 SCP 코드 분포 — 진단 라벨과 별개 축이라 여태 안 걸렀다")
for i, c in enumerate(CLASSES):
    m = Y == i
    cnt = Counter(k for j in np.where(m)[0] for k in rl[j])
    top = ", ".join(f"{k} {v}" for k, v in cnt.most_common(5)) or "(없음)"
    run.log(f"  {c:5s} n={int(m.sum()):5,} · 순수SR {int(is_sr[m].sum()):5,}"
            f" ({is_sr[m].mean()*100:5.1f}%) · 리듬코드없음 {int(no_rhy[m].sum()):4,} · {top}")

run.log("\n── (b) 순수 SR 레코드만으로 Cohen's d 재계산")
oth = Y != 0
n_a, n_b = int(((~oth) & is_sr).sum()), int((oth & is_sr).sum())
if min(n_a, n_b) < 50:
    run.log(f"  ⛔ 표본 부족 (NORM {n_a} · 질환 {n_b}) — 재계산 보류")
    d2, mx2 = {}, None
else:
    d2 = {}
    for j, nm in enumerate(FEAT):
        a, b = R[(~oth) & is_sr, j], R[oth & is_sr, j]
        d2[nm] = float((a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2 + 1e-9))
    mx2 = max(abs(v) for v in d2.values())
    run.log(f"  대상 NORM {n_a:,} · 질환 {n_b:,}")
    for nm in FEAT:
        run.log(f"  {nm:10s} d={pre[nm]:+.3f} → {d2[nm]:+.3f}  (Δ{d2[nm]-pre[nm]:+.3f})")
    run.log(f"  최대 |d| {mx:.3f} → {mx2:.3f}")

run.log("\n── (c) R검출 건전성 (CD의 넓은 QRS가 검출을 흔드는가)")
fail  = R[:, 5] == 0      # 최단/중앙==0 → rr_features가 0벡터 반환(피크 4개 미만)
tachy = R[:, 1] > 120     # 이중검출이면 심박수가 두 배로 뜬다
for i, c in enumerate(CLASSES):
    m = Y == i
    run.log(f"  {c:5s} 검출실패 {fail[m].mean()*100:5.2f}% · "
            f"심박수>120 {tachy[m].mean()*100:5.2f}% · "
            f"심박수 중앙 {np.median(R[m, 1]):5.1f}")

# ── 판정 (★ 전체 최대값 하나로 판정하지 않는다)
#   7개 특징은 사실 두 덩어리다. 심박수 계열이 유지되면서 불규칙 계열만 무너지는 게
#   가장 흔한 그림인데, 전체 최대값만 보면 그 붕괴가 통째로 가려진다.
GROUP = {"심박수 계열": ["RR중앙", "심박수"],
         "불규칙 계열": ["RR산포", "RMSSD", "pNN50", "최단/중앙", "최장/중앙"]}

def gmax(dd, keys):
    return max(abs(dd[k]) for k in keys)

run.log("\n" + "=" * 78)
verdict = {}
if mx2 is None:
    run.log("판정 보류 — 표본 부족")
else:
    for g, keys in GROUP.items():
        b, a = gmax(pre, keys), gmax(d2, keys)
        drop = 1 - a / (b + 1e-9)
        if a < 0.30 and drop > 0.3:
            v = "오염"; msg = "리듬코드를 거르니 무너짐 → 같이 붙은 부정맥을 본 것"
        elif a >= 0.30 and drop <= 0.3:
            v = "유지"; msg = "순수 SR에서도 살아남음 → 오염이 아니다"
        elif a < 0.30:
            v = "무의미"; msg = "애초에 판별력이 없음(사전등록 예측대로)"
        else:
            v = "혼합"; msg = "일부만 무너짐"
        verdict[g] = {"before": b, "after": a, "verdict": v}
        run.log(f"  {g}: |d| {b:.3f} → {a:.3f} (잔존 {a/(b+1e-9)*100:3.0f}%)  **{v}** — {msg}")

    run.log("")
    rate, irr = verdict["심박수 계열"], verdict["불규칙 계열"]
    if rate["verdict"] == "유지" and irr["verdict"] in ("오염", "무의미"):
        run.log("→ **(b) 중증도 대리가 본체**. 남은 건 '아프면 맥이 빠르다' 하나다.")
        run.log("   기전이 아니라 비특이 지름길 — 코호트를 옮기면 깨진다.")
        run.log("   R arm이 점수를 내도 '리듬 축이 전도장애를 안다'는 뜻이 절대 아니다.")
    elif irr["verdict"] == "유지":
        run.log("→ **불규칙 축이 순수 SR에서도 살아남았다**. 오염으로 설명이 안 된다.")
        run.log("   (c) 검출 아티팩트를 배제해야 한다 — 위 검출실패율·빈맥률을 볼 것.")
    elif rate["verdict"] in ("오염", "무의미") and irr["verdict"] in ("오염", "무의미"):
        run.log("→ **(a) 라벨 오염이 주범**. 사전등록 예측(리듬은 못 가른다)이 사실상 맞았다.")
    else:
        run.log("→ **혼합** — 두 계열의 판정이 엇갈린다. 결과 카드에 둘 다 적는다.")
run.log("=" * 78)
run.log("어느 쪽이든 CELL 6은 그대로 진행 — 셋 다 R을 부풀려 M−R을 **작게** 만드는 방향이다.")

run.save_json("precheck_rhythm_diag", {
    "cohens_d_all": pre, "max_abs_d_all": mx,
    "cohens_d_sr_only": d2, "max_abs_d_sr_only": mx2, "group_verdict": verdict,
    "n_sr_norm": n_a, "n_sr_disease": n_b,
    "rhythm_code_counts": {CLASSES[i]: dict(Counter(
        k for j in np.where(Y == i)[0] for k in rl[j])) for i in range(3)},
    "pure_sr_rate": {CLASSES[i]: float(is_sr[Y == i].mean()) for i in range(3)},
    "detect_fail_rate": {CLASSES[i]: float(fail[Y == i].mean()) for i in range(3)},
    "tachy_rate": {CLASSES[i]: float(tachy[Y == i].mean()) for i in range(3)},
})

In [ ]:
# CELL 6 — arm 학습: M(형태만) / R(리듬만) / MR(둘 다)
import tensorflow as tf
from tensorflow.keras import layers, models

tr = np.isin(FOLD, range(1, 9)); va = FOLD == 9; te = FOLD == 10
run.log(f"학습 {tr.sum():,} / 검증 {va.sum():,} / 테스트 {te.sum():,}  "
        f"(공식 fold, 환자 계층화)")
run.log(f"테스트 클래스 {np.bincount(Y[te], minlength=3).tolist()}")

def auto_weights(y, beta=0.9999):
    w = {c: (1 - beta) / (1 - beta ** max((y == c).sum(), 1)) for c in range(3)}
    return {c: float(v / w[0]) for c, v in w.items()}
CW = auto_weights(Y[tr]); SW = np.array([CW[int(c)] for c in Y[tr]], "float32")
run.log(f"가중치 { {CLASSES[c]: round(v,2) for c,v in CW.items()} } · "
        f"가중 후 { {CLASSES[c]: int((Y[tr]==c).sum()*CW[c]) for c in range(3)} }")

def norm(F, m):
    md_ = np.median(F[m], 0)
    iq = np.percentile(F[m], 75, 0) - np.percentile(F[m], 25, 0)
    return np.clip((F - md_) / (iq + 1e-3), -10, 10).astype("float32")
Rn = norm(R, tr)
Xf = X[..., None]

def build(kind, seed):
    tf.keras.utils.set_random_seed(seed)
    ins, feats = [], []
    if kind in ("M", "MR"):
        si = layers.Input((X.shape[1], 1)); x = si
        for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
            x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
            x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
        x = layers.GlobalAveragePooling1D()(x)
        ins.append(si); feats.append(layers.Dense(64, activation="relu")(x))
    if kind in ("R", "MR"):
        fi = layers.Input((R.shape[1],))
        g = layers.Dense(64, activation="relu")(fi)
        g = layers.Dense(64, activation="relu")(g)
        ins.append(fi); feats.append(g)
    h = feats[0] if len(feats) == 1 else layers.Concatenate()(feats)
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(ins, layers.Dense(3, activation="softmax")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="sparse_categorical_crossentropy")
    return m

def pack(kind, m):
    if kind == "M":  return Xf[m]
    if kind == "R":  return Rn[m]
    return [Xf[m], Rn[m]]

probs, t0 = {}, time.time()
for kind in ("M", "R", "MR"):
    c = run.load_arm(kind)
    if c is not None:
        probs[kind] = c; run.log(f"⏭ {kind} 이미 완료"); continue
    acc = np.zeros((int(te.sum()), 3))
    for s in range(N_SEEDS):
        m = build(kind, SEED0 + s)
        h = m.fit(pack(kind, tr), Y[tr], validation_data=(pack(kind, va), Y[va]),
                  epochs=EPOCHS, batch_size=128, sample_weight=SW, verbose=0)
        pr = m.predict(pack(kind, te), batch_size=512, verbose=0)
        acc += pr
        run.log(f"  {kind} seed {s+1}/{N_SEEDS} ({time.time()-t0:.0f}s) "
                f"loss {h.history['loss'][0]:.3f}→{h.history['loss'][-1]:.3f} "
                f"val {h.history['val_loss'][-1]:.3f} | "
                f"예측분포 {np.bincount(pr.argmax(1), minlength=3).tolist()}")
        if s == 0:
            run.save_model(m, kind)
        tf.keras.backend.clear_session()
    probs[kind] = acc / N_SEEDS
    run.save_arm(kind, probs[kind])

In [ ]:
# CELL 7 — 평가 + 사전등록 판정
from sklearn.metrics import f1_score, confusion_matrix, classification_report
yte = Y[te]
res = {}
for k, pr in probs.items():
    p = pr.argmax(1)
    res[k] = {"pred": p, "macro": float(f1_score(yte, p, average="macro", zero_division=0)),
              "f1": f1_score(yte, p, average=None, labels=[0, 1, 2], zero_division=0),
              "cm": confusion_matrix(yte, p, labels=[0, 1, 2]).tolist()}

NAME = {"M": "형태만(CNN)", "R": "리듬만(RR요약)", "MR": "형태+리듬"}
# ★ 실험2의 교훈: 주지표 하나로 판정하지 않는다. 오경보율을 함께 본다.
#   웨어러블/스크리닝에서 최대 실패는 오진이 아니라 오경보(alarm fatigue)다.
norm_m = (yte == 0)
for k, r in res.items():
    r["false_alarm"] = float((r["pred"][norm_m] != 0).mean())   # 정상인데 병으로
    r["miss"] = float((r["pred"][~norm_m] == 0).mean())         # 병인데 정상으로

run.log("=" * 96)
run.log(f"{'ARM':<22}{'macro-F1':>10}" + "".join(f"{c:>9}" for c in CLASSES) +
        f"{'오경보율':>10}{'놓침율':>9}")
run.log("=" * 96)
for k in ("M", "R", "MR"):
    r = res[k]
    run.log(f"{k+' '+NAME[k]:<22}{r['macro']:>10.4f}" +
            "".join(f"{r['f1'][i]:>9.3f}" for i in range(3)) +
            f"{r['false_alarm']:>10.3f}{r['miss']:>9.3f}")
run.log("=" * 96)
run.log("  오경보율 = 정상(NORM)인데 병으로 예측한 비율 · 놓침율 = 병인데 정상으로 예측한 비율")

COLLAPSED = [k for k, r in res.items() if (r["f1"][1] == 0 and r["f1"][2] == 0)]
if COLLAPSED:
    run.log(f"⛔ 붕괴 arm: {COLLAPSED} — 판정 보류")

def boot_f1(pa, pb, B=CONFIG["boot"], seed=SEED0):
    """레코드 단위 부트스트랩으로 macro-F1 차이의 CI."""
    rs = np.random.RandomState(seed); n = len(yte); ds = []
    for _ in range(B):
        i = rs.randint(0, n, n)
        ds.append(f1_score(yte[i], pa[i], average="macro", zero_division=0) -
                  f1_score(yte[i], pb[i], average="macro", zero_division=0))
    ds = np.array(ds)
    return float(ds.mean()), float(np.percentile(ds, 2.5)), float(np.percentile(ds, 97.5))

boots = {}
run.log(f"\n{'비교':<26}{'Δ':>10}{'95% CI':>24}   판정")
run.log("-" * 74)
for hi, lo in (("M", "R"), ("MR", "M"), ("MR", "R")):
    d, l, u = boot_f1(res[hi]["pred"], res[lo]["pred"])
    sig = bool(l > 0 or u < 0)
    boots[f"{hi}-{lo}"] = {"delta": d, "ci": [l, u], "significant": sig}
    run.log(f"{NAME[hi]+' − '+NAME[lo]:<26}{d:>+10.4f}   [{l:+.4f}, {u:+.4f}]   "
            f"{'유의 ★' if sig else '비유의'}")
run.log("-" * 74)

# ── 동작점 정합 비교(사후, 사전등록 아님) ──────────────────────────
#   실험2에서 argmax 비교가 '성능 변화'와 '동작점 변화'를 섞는다는 걸 확인했다.
#   NORM 확률에 계수를 곱해 같은 오경보율로 맞춘 뒤 다시 비교한다.
def at_same_false_alarm(prob, target_fa):
    lo, hi = 0.02, 50.0
    for _ in range(40):
        mid = (lo * hi) ** 0.5
        p = prob.copy(); p[:, 0] *= mid
        fa = float((p.argmax(1)[norm_m] != 0).mean())
        if fa > target_fa:
            lo = mid
        else:
            hi = mid
    p = prob.copy(); p[:, 0] *= hi
    return p.argmax(1), hi

ref_fa = res["M"]["false_alarm"]
run.log(f"\n[사후·사전등록 아님] 오경보율을 M 기준({ref_fa:.3f})으로 맞춘 뒤 비교")
for k in ("R", "MR"):
    pk, alpha = at_same_false_alarm(probs[k], ref_fa)
    f1k = f1_score(yte, pk, average="macro", zero_division=0)
    run.log(f"  {k:<4} macro-F1 {res[k]['macro']:.4f}(argmax) → {f1k:.4f}(동작점 정합) "
            f"· α={alpha:.2f} · 오경보율 {float((pk[norm_m]!=0).mean()):.3f}")

mr, mm = boots["M-R"], boots["MR-M"]
run.log(f"\n▶ 사전등록 주가설 — 형태(M) − 리듬(R)")
run.log(f"   Δ = {mr['delta']:+.4f}  CI [{mr['ci'][0]:+.4f}, {mr['ci'][1]:+.4f}]")
run.log(f"   [대조] 실험1′(부정맥)에서는 리듬이 형태를 이겼다: B3−B2 Δ+0.1268")
run.log(f"▶ 보조 — 리듬을 더하면 오르나 (MR − M): Δ={mm['delta']:+.4f} "
        f"[{mm['ci'][0]:+.4f}, {mm['ci'][1]:+.4f}]")

if COLLAPSED:
    verdict = f"판정 불가 — arm 붕괴({COLLAPSED})"
elif mr["significant"] and mr["delta"] > 0:
    verdict = ("확증 — 전도장애·ST-T는 형태 축이 담당한다. 실험1′(부정맥=리듬 축)과 "
               "합쳐 **두 축이 서로 다른 질환군을 담당함이 양방향으로 증명**됨")
elif mr["significant"] and mr["delta"] < 0:
    verdict = "예측 반전 — 리듬이 형태를 이겼다. 라벨에 부정맥이 섞였는지 확인 필요"
else:
    verdict = "미결 — 두 축의 차이를 검출하지 못함(검정력 또는 과제 난이도)"
run.log(f"   → {verdict}")

run.save_json("evaluation", {
    "arms": {k: {"macro_f1": r["macro"],
                 "f1_per_class": {CLASSES[i]: float(r["f1"][i]) for i in range(3)},
                 "confusion": r["cm"]} for k, r in res.items()},
    "bootstrap": boots, "collapsed": COLLAPSED, "verdict": verdict,
    "precheck_rhythm_max_d": mx})

result = {"week": 2, "exp_id": "exp3_ptbxl", "quest": "ailab-2026-0015",
          "task": "PTB-XL lead II — 형태 축 vs 리듬 축 (전도장애·ST-T)",
          "split": "inter", "metric": "macro_f1",
          "value": round(res["MR"]["macro"], 4),
          "passed": bool(mr["significant"] and mr["delta"] > 0 and not COLLAPSED),
          "date": time.strftime("%Y-%m-%d"),
          "arms": {k: round(r["macro"], 4) for k, r in res.items()},
          "bootstrap": boots, "verdict": verdict,
          "summary": f"M-R {mr['delta']:+.4f} [{mr['ci'][0]:+.4f},{mr['ci'][1]:+.4f}] "
                     f"MR-M {mm['delta']:+.4f} → {verdict.split(' —')[0]}"}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp3_ptbxl_morph_vs_rhythm.ipynb \\
      --quest ailab-2026-0015 --step "exp3-ptbxl-morph-vs-rhythm" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")
print(classification_report(yte, res["MR"]["pred"], target_names=CLASSES, digits=3))

---

## 결과 읽는 법

| M − R | 뜻 | 다음 |
|---|---|---|
| **크게 양수 + MR−M ≈ 0** | **예측 적중.** 형태 축과 리듬 축이 서로 다른 질환군을 담당한다는 게 양방향 증명 | Finding Head의 축 분리를 확정. 실험 10(유도 확장)으로 |
| 양수지만 MR−M도 양수 | 두 축이 **부분적으로 상보** | 축 분리보다 **융합**이 맞다는 뜻 — 구조 재검토 |
| **음수(리듬이 이김)** | 라벨에 부정맥이 섞였을 가능성 | CELL 5·5.5의 진단 결과를 다시 볼 것 |
| **M−R 0 근처** | 결론 불가 — R이 기전으로 번 건지 지름길로 번 건지 모른다 | CELL 5.5가 (b)면 지름길 쪽에 무게 |
| 붕괴 | 학습 실패 | 가중치·에폭 조정 후 재실행 |

## 이 실험이 남기는 것
1. **PTB-XL lead II 파이프라인** — 실험 10(유도 ablation)에서 `{I,II}`·`{II,V1}`·`{12}`로
   슬라이스만 바꾸면 바로 돌아간다
2. **라벨-유도 정합성 필터의 첫 실천** — MI·HYP를 뺀 근거가 코드에 남는다
3. **부정맥 → 진단으로 넘어간 첫 기록**
